In [1]:
import glob
import pandas as pd
import os

# Find all Excel files in the current directory
excel_files = glob.glob('*.xlsx') + glob.glob('*.xls')

if not excel_files:
    print("No Excel files found in the current folder.")
else:
    print(f"Found {len(excel_files)} Excel file(s): {excel_files}")

    # List to store dataframes
    df_list = []

    for file in excel_files:
        try:
            # Read each Excel file
            df = pd.read_excel(file)
            # Optional: Add a column to keep track of the source file
            df['source_file'] = file
            df_list.append(df)
            print(f"Successfully loaded: {file}")
        except Exception as e:
            print(f"Error reading {file}: {e}")

    if df_list:
        # Concatenate all dataframes
        compiled_df = pd.concat(df_list, ignore_index=True)

        compiled_df = compiled_df.sort_values(by='Tanggal', key=lambda col: pd.to_datetime(col, format='ISO8601'))
        # # Save the compiled dataframe to a new Excel file
        # output_filename = 'compiled_excel_output.xlsx'
        # compiled_df.to_excel(output_filename, index=False)
        # print(f"\nAll files compiled successfully! Saved as: {output_filename}")

        # Display the first few rows of the compiled data
        display(compiled_df.head())
    else:
        print("No data could be compiled.")

Found 4 Excel file(s): ['Investor_2026-11-11_22-55-05.xlsx', 'Investor_2026-11-12_22-50-00.xlsx', 'Investor_2026-11-14_22-30-08.xlsx', 'Investor_2026-11-15_22-32-25.xlsx']
Successfully loaded: Investor_2026-11-11_22-55-05.xlsx
Successfully loaded: Investor_2026-11-12_22-50-00.xlsx
Successfully loaded: Investor_2026-11-14_22-30-08.xlsx
Successfully loaded: Investor_2026-11-15_22-32-25.xlsx


,Judul,Detail,Tanggal,Link,Kategori,Source,Crawled Date,source_file
188,"Pendapatan Anabatic (ATIC) Tumbuh 17,8%","JAKARTA, Investor.id– PT Anabatic Technologies...",2026-11-10 15:10:00,https://investor.id/market/461416/pendapatan-a...,Market,Investor,2026-11-11 22:55:05,Investor_2026-11-11_22-55-05.xlsx
187,"Telkom (TLKM) Rampingkan Anak Usaha, 2 Entitas...","JAKARTA, Investor.id- PT Telkom Indonesia (Per...",2026-11-10 15:30:00,https://investor.id/market/461417/telkom-tlkm-...,Market,Investor,2026-11-11 22:55:05,Investor_2026-11-11_22-55-05.xlsx
186,Adaro Strategic Investments Alihkan Seluruh Sa...,"JAKARTA, investor.id– PT Alamtri Resources Ind...",2026-11-10 15:39:00,https://investor.id/market/461418/adaro-strate...,Market,Investor,2026-11-11 22:55:05,Investor_2026-11-11_22-55-05.xlsx
185,"Gibran Ajukan Pengunduran Diri, Istana: Prabow...","JAKARTA, investor.id– Wakil Presiden (Wapres) ...",2026-11-10 15:42:00,https://investor.id/national/461419/gibran-aju...,National,Investor,2026-11-11 22:55:05,Investor_2026-11-11_22-55-05.xlsx
184,Singapura Luncurkan Layanan Brankas Emas Khusu...,"SINGAPURA, investor.id –Singapura resmi melunc...",2026-11-10 15:57:00,https://investor.id/international/461420/singa...,International,Investor,2026-11-11 22:55:05,Investor_2026-11-11_22-55-05.xlsx


In [2]:
import pandas as pd

# Ensure the 'Tanggal' column is converted to datetime format
compiled_df['Tanggal_parsed'] = pd.to_datetime(compiled_df['Tanggal'], errors='coerce')

# Extract the day of the week (Monday=0, Tuesday=1, Wednesday=2, Thursday=3, Friday=4, Saturday=5, Sunday=6)
compiled_df['day_of_week'] = compiled_df['Tanggal_parsed'].dt.dayofweek

# Group 1: Thursday (3) to Sunday (6)
category_thu_sun = compiled_df[compiled_df['day_of_week'].isin([3, 4, 5, 6])].copy()

# Group 2: Monday (0) to Wednesday (2)
# category_mon_wed = compiled_df[compiled_df['day_of_week'].isin([0, 1, 2])].copy()

# Drop helper columns from output
for df_cat in [category_thu_sun]:
    df_cat.drop(columns=['Tanggal_parsed', 'day_of_week', 'source_file'], inplace=True, errors='ignore')

# # Save the filtered categories to separate Excel files
# category_thu_sun.to_excel('news_thursday_to_sunday.xlsx', index=False)
# category_mon_wed.to_excel('news_monday_to_wednesday.xlsx', index=False)

# print(f"Category 1 (Thursday - Sunday): {len(category_thu_sun)} rows saved to 'news_thursday_to_sunday.xlsx'")
# print(f"Category 2 (Monday - Wednesday): {len(category_mon_wed)} rows saved to 'news_monday_to_wednesday.xlsx'")

# Display previews
print("\n--- Preview: Thursday to Sunday ---")
display(category_thu_sun.head(2))

# print("\n--- Preview: Monday to Wednesday ---")
# display(category_mon_wed.head(2))


--- Preview: Thursday to Sunday ---


,Judul,Detail,Tanggal,Link,Kategori,Source,Crawled Date
5,"Harga Emas Gagal Bertahan di US$ 4.000, Ini Pe...","NEW YORK, investor.id– Harga emas dunia kembal...",2026-11-12 04:14:00,https://investor.id/market/461622/harga-emas-g...,Market,Investor,2026-11-11 22:55:05
355,"Harga Emas Gagal Bertahan di US$ 4.000, Ini Pe...","NEW YORK, investor.id– Harga emas dunia kembal...",2026-11-12 04:14:00,https://investor.id/market/461622/harga-emas-g...,Market,Investor,2026-11-12 22:50:00


In [3]:
category_thu_sun = category_thu_sun.drop_duplicates(subset=['Detail'], keep='last')

from datetime import datetime

# 1. Get current system date and time
now = datetime.now()

# 2. Format it safely for a filename (Year-Month-Day_Hour-Minute-Second)
timestamp = now.strftime("%Y-%m-%d_%H-%M-%S")

In [4]:
import os

# Define the name of the new folder
target_folder = 'categorized_news'

# Create the folder if it does not exist
os.makedirs(target_folder, exist_ok=True)

# Define the target file paths
thu_sun_path = os.path.join(target_folder, f'news_thursday_to_sunday_{timestamp}.xlsx')
# mon_wed_path = os.path.join(target_folder, f'news_monday_to_wednesday_{timestamp}.xlsx')

# Save the filtered dataframes into the new folder
category_thu_sun.to_excel(thu_sun_path, index=False)
# category_mon_wed.to_excel(mon_wed_path, index=False)

print(f"Created directory: '{target_folder}'")
print(f"Successfully saved Category 1 to: '{thu_sun_path}'")
# print(f"Successfully saved Category 2 to: '{mon_wed_path}'")

Created directory: 'categorized_news'
Successfully saved Category 1 to: 'categorized_news/news_thursday_to_sunday_2026-11-16_08-59-08.xlsx'
